# 06 — CAMCGE: Reproducing a Published CGE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/06_camcge_replication.ipynb)

CGE-Core's `cam/` benchmark implements the Cameroon model published by **Condon, Dahl, and Devarajan (1987)** and compares the Pyomo solution with the published numerical results.

The question is now:

> **Can the software reproduce an independently published CGE benchmark?**

The CAMCGE code intentionally lives outside the installed `cge_core` package because it is a replication/regression benchmark.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

WORKSPACE = Path("/content") if Path("/content").exists() else Path.home() / ".cache"
WORKSPACE.mkdir(parents=True, exist_ok=True)
REPO_DIR = WORKSPACE / "CGE-core-colab"

if REPO_DIR.exists():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "fetch", "origin", "main", "--depth", "1"],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"],
        check=True,
    )
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/miraflor/CGE-core.git", str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

import cge_core
print("✓ CGE-Core", cge_core.__version__)
print("✓ Repository:", REPO_DIR)


subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "amplpy"],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "amplpy.modules", "install", "coin"],
    check=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

module_path = subprocess.check_output(
    [sys.executable, "-m", "amplpy.modules", "path"],
    text=True,
).strip()
os.environ["PATH"] = module_path + os.pathsep + os.environ.get("PATH", "")

assert shutil.which("ipopt"), "IPOPT was not found after installing the COIN module."
SOLVER = "ipopt"
print("✓ Solver:", SOLVER)

## 2. Reproduce the published base equilibrium

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pyomo.environ import value

from cam.replicate_base import (
    LEVEL_GROUPS,
    PUB,
    base_metrics,
    build_base,
    validate_base,
)

cge, dof_before, dof_after = build_base(SOLVER)
metrics = base_metrics(cge, dof_before, dof_after)
validate_base(metrics)

display(
    pd.Series(
        {
            "DOF before redundant equation drop": metrics["dof_before"],
            "DOF after drop": metrics["dof_after"],
            "Solved omega": metrics["omega"],
            "Published omega": PUB["omega"],
            "|omega difference|": metrics["omega_difference"],
            "Published levels checked": metrics["published_level_count"],
            "Worst |level - published|": metrics["worst_level_difference"],
            "Dropped current-account gap": metrics["current_account_gap"],
        },
        name="CAMCGE base replication",
    ).to_frame()
)

## 3. Inspect the 98 published variable-level comparisons

In [ ]:
rows = []

for variable_name, index in LEVEL_GROUPS:
    variable = getattr(cge.base, variable_name)
    for item, published in zip(index, PUB[variable_name]):
        solved = value(variable[item])
        rows.append({
            "variable": variable_name,
            "index": item,
            "solved": solved,
            "published": published,
            "difference": solved - published,
            "abs_difference": abs(solved - published),
        })

level_comparison = pd.DataFrame(rows).sort_values("abs_difference", ascending=False)

print("Largest numerical discrepancies:")
display(
    level_comparison.head(15).style.format({
        "solved": "{:.6f}",
        "published": "{:.6f}",
        "difference": "{:+.6f}",
        "abs_difference": "{:.2e}",
    })
)

The validation also checks that the deliberately dropped current-account equation clears at the solved equilibrium.

## 4. Replicate a published policy experiment

In [ ]:
from cam.replicate_experiments import experiment_3, snapshot

base_snapshot = snapshot(cge.base)

# Experiment 3: double tariffs on intermediate goods
# and construction materials.
exp3 = experiment_3(cge, base_snapshot, SOLVER)

display(
    pd.Series(
        {
            "Tariff revenue, % change": exp3["tariff_revenue"],
            "Real investment, % change": exp3["investment"],
            "Imports of biens-int, % change": exp3["dm_biens_int"],
            "Imports of cim-int, % change": exp3["dm_cim_int"],
        },
        name="Experiment 3",
    ).to_frame()
)

## 5. See the sectoral pattern

In [ ]:
output = pd.Series(exp3["dxd"], name="Output")
prices = pd.Series(exp3["dp"], name="Composite price")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(output.index, output.values)
ax.axhline(0, linewidth=0.8)
ax.set_ylabel("% change")
ax.set_title("CAMCGE Experiment 3 — sector output")
ax.tick_params(axis="x", rotation=45)
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(prices.index, prices.values)
ax.axhline(0, linewidth=0.8)
ax.set_ylabel("% change")
ax.set_title("CAMCGE Experiment 3 — composite prices")
ax.tick_params(axis="x", rotation=45)
plt.show()

## Why this notebook matters

A replication benchmark tests **implementation fidelity and reproducibility**, not merely whether a solver returns a solution.

## Next

The final notebook opens the machine: Pyomo objects, closure, degrees of freedom, calibration parameters, and Walras' law.

[Open Notebook 07 in Colab](https://colab.research.google.com/github/miraflor/CGE-core/blob/main/notebooks/07_under_the_hood.ipynb)